# Elhub Bronze Municipality Consumption

In [1]:
import json

# Make imports work whether Jupyter starts in the repository root or notebooks/.
import sys
import zipfile
from pathlib import Path

import pandas as pd

project_root = Path.cwd().resolve()
while (
    not (project_root / "bronze_layer").is_dir() and project_root != project_root.parent
):
    project_root = project_root.parent
if not (project_root / "bronze_layer").is_dir():
    raise RuntimeError("Start Jupyter from inside the project repository")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from bronze_layer.elhub import ingest_dataset, inspect_zip

print("Project root:", project_root)

Project root: /home/roland/UiS/rewrite/group14


## Ingest

Set `refresh=True` to request the latest Elhub snapshot.

In [2]:
DATASET = "CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR"
BRONZE_ROOT = project_root / "data/bronze/elhub"
result = ingest_dataset(DATASET, root=BRONZE_ROOT, refresh=False)
result

CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR: cached (37 CSV files)


{'dataset': 'CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR',
 'status': 'cached',
 'archive_path': '/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip',
 'metadata_path': '/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip.metadata.json',
 'csv_count': 37,
 'sha256': '2a7be0d7e979bb07401eb5ce0b93d73d99f7064e106df9d9ab2ee52f12886add'}

## Inspect archive and metadata

In [3]:
archive_path = Path(result["archive_path"])
metadata_path = Path(result["metadata_path"])
csv_files = inspect_zip(archive_path)
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
print("Archive:", archive_path.name)
print("Size (MiB):", round(archive_path.stat().st_size / 1024**2, 2))
print("CSV files:", len(csv_files))
print("SHA-256:", metadata["sha256"])
print(
    "Advertised period:",
    metadata["advertised_period_start"],
    "to",
    metadata["advertised_period_end"],
)
print("Period verified from CSV contents:", metadata["period_verified"])
print("First CSV filenames:")
for name in csv_files[:10]:
    print(" -", name)

Archive: CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip
Size (MiB): 427.52
CSV files: 37
SHA-256: 2a7be0d7e979bb07401eb5ce0b93d73d99f7064e106df9d9ab2ee52f12886add
Advertised period: 2023-11-17 to 2026-11-17
Period verified from CSV contents: False
First CSV filenames:
 - part-00000-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00001-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00002-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00003-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00004-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00005-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00006-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00007-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00008-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
 - part-00009-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv


## Preview a CSV as a table

Read a limited number of rows directly from the ZIP without extracting or modifying Bronze.

In [4]:
csv_index = 0
preview_rows = 20
with zipfile.ZipFile(archive_path) as archive:
    with archive.open(csv_files[csv_index]) as file:
        df = pd.read_csv(
            file, sep=None, engine="python", encoding="utf-8-sig", nrows=preview_rows
        )
print("File:", csv_files[csv_index])
print("Columns:", df.columns.tolist())
display(df.head(10))

File: part-00000-a37d65b4-e38f-4a53-b2ca-8f8507186f40-c000.csv
Columns: ['START_TIME', 'END_TIME', 'MUNICIPALITY_NUMBER', 'NAME', 'CONSUMPTION_GROUP', 'QUANTITY_KWH', 'METERING_POINT_COUNT', 'LAST_UPDATE_TIME']


,START_TIME,END_TIME,MUNICIPALITY_NUMBER,NAME,CONSUMPTION_GROUP,QUANTITY_KWH,METERING_POINT_COUNT,LAST_UPDATE_TIME
0,2023-11-17T00:00:00.000+01:00,2023-11-17T01:00:00.000+01:00,0,Ukjent tilhørighet,business,1654.807,171,2026-02-13T03:20:12.524+01:00
1,2023-11-17T01:00:00.000+01:00,2023-11-17T02:00:00.000+01:00,0,Ukjent tilhørighet,business,1640.203,171,2026-02-13T03:20:12.524+01:00
2,2023-11-17T02:00:00.000+01:00,2023-11-17T03:00:00.000+01:00,0,Ukjent tilhørighet,business,1728.473,171,2026-02-13T03:20:12.524+01:00
3,2023-11-17T03:00:00.000+01:00,2023-11-17T04:00:00.000+01:00,0,Ukjent tilhørighet,business,1659.915,171,2026-02-13T03:20:12.524+01:00
4,2023-11-17T04:00:00.000+01:00,2023-11-17T05:00:00.000+01:00,0,Ukjent tilhørighet,business,1655.694,171,2026-02-13T03:20:12.524+01:00
5,2023-11-17T05:00:00.000+01:00,2023-11-17T06:00:00.000+01:00,0,Ukjent tilhørighet,business,1747.511,171,2026-02-13T03:20:12.524+01:00
6,2023-11-17T06:00:00.000+01:00,2023-11-17T07:00:00.000+01:00,0,Ukjent tilhørighet,business,2109.402,171,2026-02-13T03:20:12.524+01:00
7,2023-11-17T07:00:00.000+01:00,2023-11-17T08:00:00.000+01:00,0,Ukjent tilhørighet,business,2351.958,171,2026-02-13T03:20:12.524+01:00
8,2023-11-17T08:00:00.000+01:00,2023-11-17T09:00:00.000+01:00,0,Ukjent tilhørighet,business,2410.283,171,2026-02-13T03:20:12.524+01:00
9,2023-11-17T09:00:00.000+01:00,2023-11-17T10:00:00.000+01:00,0,Ukjent tilhørighet,business,2377.702,171,2026-02-13T03:20:12.524+01:00


## Check types in the preview

In [5]:
display(
    pd.DataFrame({"dtype": df.dtypes.astype(str), "nulls_in_preview": df.isna().sum()})
)

,dtype,nulls_in_preview
START_TIME,str,0
END_TIME,str,0
MUNICIPALITY_NUMBER,int64,0
NAME,str,0
CONSUMPTION_GROUP,str,0
QUANTITY_KWH,float64,0
METERING_POINT_COUNT,int64,0
LAST_UPDATE_TIME,str,0


## Check cached files

In [6]:
for path in sorted((BRONZE_ROOT / DATASET).iterdir()):
    print(path.name)

CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip
CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip.metadata.json
_runs
